# Retrieval-Augmented Generation (RAG) with DistLLM

This notebook demonstrates a minimal, self-contained RAG pipeline using the DistLLM OpenAI-compatible API:

1. Define a handful of sample documents.
2. Embed them with the DistLLM **embeddings** endpoint.
3. Build a tiny in-notebook vector store (cosine similarity with numpy + sklearn).
4. Retrieve the top-k most relevant chunks for a question.
5. Generate an answer with a **chat completion** grounded in the retrieved context.

No external vector database is required — the index lives entirely in memory.

## Requirements

```bash
pip install openai distllm_sdk numpy scikit-learn
```

Then start the DistLLM API server (see the README in this folder).

In [ ]:
# ---------------------------------------------------------------------------
# Connection configuration
# ---------------------------------------------------------------------------
# Point the OpenAI-compatible client at a running DistLLM API.
# Start the server first, e.g.:
#     distllm-api --model distributed-llm --host 0.0.0.0 --port 8000
#
# The API key is not required for a local deployment; set DISTLLM_API_KEY if
# your deployment enforces authentication.
import os

BASE_URL = os.environ.get("DISTLLM_BASE_URL", "http://localhost:8000/v1")
API_KEY = os.environ.get("DISTLLM_API_KEY", "not-needed")
MODEL = os.environ.get("DISTLLM_MODEL", "distributed-llm")

print(f"Using endpoint: {BASE_URL}")
print(f"Model: {MODEL}")


In [ ]:
# ---------------------------------------------------------------------------
# 1. Sample documents
# ---------------------------------------------------------------------------
# A small corpus about DistLLM. Swap these for your own text to adapt the demo.
documents = [
    "DistLLM is a distributed inference system that spreads large language model "
    "workloads across many small GPUs instead of requiring a single giant accelerator.",
    "The coordinator node routes requests and aggregates token streams from worker "
    "nodes that each hold a shard of the model weights.",
    "DistLLM exposes an OpenAI-compatible REST API so existing OpenAI client code "
    "keeps working with only a changed base_url.",
    "Embeddings in DistLLM are produced by a shared encoder and can be cached to "
    "speed up retrieval across repeated queries.",
    "Cost tracking is built in: every response carries usage fields such as "
    "prompt_tokens, completion_tokens and an estimated cost_usd.",
    "Speculative decoding lets a small draft model propose tokens that a larger "
    "target model verifies, improving throughput on commodity hardware.",
]

print(f'Loaded {len(documents)} documents.')


In [ ]:
# ---------------------------------------------------------------------------
# 2. Embed the documents via the DistLLM embeddings endpoint
# ---------------------------------------------------------------------------
# We prefer the official 'openai' client pointed at the DistLLM /v1 endpoint.
# If it is not installed we transparently fall back to distllm_sdk.compat.
try:
    from openai import OpenAI
    client = OpenAI(base_url=BASE_URL, api_key=API_KEY)
    print('Using official openai client.')
except ImportError:
    from distllm_sdk.compat import openai_compat as oc
    client = oc.OpenAI(base_url=BASE_URL, api_key=API_KEY)
    print('openai client missing -> using distllm_sdk.compat.openai_compat.')

try:
    emb_resp = client.embeddings.create(model=MODEL, input=documents)
    doc_embeddings = [d.embedding for d in emb_resp.data]
    print(f'Embedded {len(doc_embeddings)} docs into vectors of dim '
          f'{len(doc_embeddings[0])}.')
except Exception as exc:  # pragma: no cover - depends on a live server
    print(f'\n[!] Embeddings request failed: {exc}')
    print('    Is the DistLLM API running at', BASE_URL, '?')
    # Provide deterministic fake embeddings so the rest of the notebook still runs.
    import numpy as _np
    _rng = _np.random.default_rng(0)
    doc_embeddings = [_rng.standard_normal(32).tolist() for _ in documents]
    print('    Falling back to random vectors for demonstration.')


In [ ]:
# ---------------------------------------------------------------------------
# 3. Build a tiny in-notebook vector store (cosine similarity)
# ---------------------------------------------------------------------------
# Optional deps are guarded so the notebook explains what is missing instead of
# crashing. We use sklearn if present, otherwise a pure-numpy fallback.
try:
    import numpy as np
    from sklearn.metrics.pairwise import cosine_similarity

    _matrix = np.asarray(doc_embeddings, dtype='float32')

    def retrieve(query, top_k=2):
        q = client.embeddings.create(model=MODEL, input=[query]).data[0].embedding
        q = np.asarray(q, dtype='float32').reshape(1, -1)
        scores = cosine_similarity(q, _matrix)[0]
        order = np.argsort(-scores)[:top_k]
        return [(int(i), float(scores[i]), documents[i]) for i in order]

    print('Vector store ready (numpy + sklearn).')
except ImportError:
    import numpy as np
    print('scikit-learn not installed -> using a pure-numpy cosine fallback.')
    _matrix = np.asarray(doc_embeddings, dtype='float32')

    def _cosine(a, b):
        return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-9))

    def retrieve(query, top_k=2):
        q = np.asarray(
            client.embeddings.create(model=MODEL, input=[query]).data[0].embedding,
            dtype='float32')
        scores = [_cosine(q, row) for row in _matrix]
        order = np.argsort(-np.array(scores))[:top_k]
        return [(int(i), float(scores[i]), documents[i]) for i in order]


In [ ]:
# ---------------------------------------------------------------------------
# 4. Retrieve top-k context for a question
# ---------------------------------------------------------------------------
question = 'How does DistLLM share model work across GPUs?'
hits = retrieve(question, top_k=2)

print('Question:', question)
print('\nRetrieved context:')
for rank, (idx, score, text) in enumerate(hits, 1):
    print(f'  [{rank}] score={score:.3f} doc#{idx}: {text}')


In [ ]:
# ---------------------------------------------------------------------------
# 5. Generate an answer grounded in the retrieved context
# ---------------------------------------------------------------------------
context = '\n\n'.join(f'- {text}' for _, _, text in hits)
messages = [
    {'role': 'system', 'content': 'Answer the user question using ONLY the provided context.'},
    {'role': 'user', 'content': f'Context:\n{context}\n\nQuestion: {question}'},
]

try:
    resp = client.chat.completions.create(
        model=MODEL, messages=messages, temperature=0.2, max_tokens=160)
    answer = resp.choices[0].message.content
    print('Answer:')
    print(answer)
    if getattr(resp, 'usage', None):
        print('\nUsage:', resp.usage)
except Exception as exc:  # pragma: no cover
    print(f'\n[!] Chat completion failed: {exc}')
    print('    Make sure the DistLLM API is running at', BASE_URL)


## Optional: LlamaIndex variant

If you prefer a higher-level retriever, install `llama-index` and the DistLLM integration (`pip install llama-index distllm_sdk`) and reuse the same `embeddings.create` call as the embedding model. The core idea — embed, store, retrieve top-k, then prompt — is identical; LlamaIndex just manages the index and the prompt assembly for you. See `examples/llamaindex_example.py` in the repo.